# Notebook 04 — Momentum Strategy

**Phase 2 · Strategy Modules (1 / 4)**

---

## 🎯 Learning Objectives

| # | Objective |
|---|----------|
| 1 | Understand the economic intuition behind momentum trading |
| 2 | Compute multi-lookback return scores |
| 3 | Build RSI, EMA, and volume **filters** to remove low-quality candidates |
| 4 | Rank assets with composite scores and apply min-max normalisation |
| 5 | Compare our scratch implementation to the production `rank_assets_by_momentum()` |

### Prerequisites
- NB01 (market data fetching)
- NB02 (RSI, EMA, Bollinger Bands)
- NB03 (Sharpe / Sortino basics for later evaluation)

In [ ]:
# ── Boilerplate ────────────────────────────────────────────
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
ROOT = str(pathlib.Path.cwd().resolve().parents[1])
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({"figure.figsize": (14, 5), "axes.grid": True})
print("✅ imports ready  |  project root:", ROOT)

---
## 1 · What Is Momentum?

**Core idea:** assets that have been going up tend to keep going up (and vice-versa) over short-to-medium horizons.  This is one of the most well-documented anomalies in finance (Jegadeesh & Titman 1993).

Our bot implements a **cross-sectional momentum** strategy:

1. Compute return scores over **multiple lookback windows** (3, 5, 7 days by default).
2. Average the lookback returns → **composite momentum score**.
3. Apply quality **filters** (RSI, EMA, volume) to remove noisy candidates.
4. **Rank** surviving assets and select the top-N.
5. **Min-max normalise** scores to [0, 1] for weight allocation.

### Why Multi-Lookback?

A single lookback is fragile.  Averaging 3-day, 5-day, and 7-day returns smooths out noise:

$$
\text{composite}(i) = \frac{1}{K} \sum_{k=1}^{K} r_{i,L_k}
\quad\text{where }\; r_{i,L_k} = \frac{P_{i,t}}{P_{i,t-L_k}} - 1
$$

With $K=3$ and $L \in \{3,5,7\}$ from `config/strategy_params.yaml`.

---
## 2 · Generating Synthetic Price Data

We'll create a realistic multi-asset price panel so every cell runs without an API key.

In [ ]:
# ── Synthetic price panel ─────────────────────────────────
np.random.seed(42)
SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "ADAUSDT",
           "XRPUSDT", "DOGEUSDT", "AVAXUSDT", "DOTUSDT",
           "LINKUSDT", "MATICUSDT", "NEARUSDT", "APTUSDT"]
DAYS = 60
dates = pd.date_range(end=pd.Timestamp.now().normalize(), periods=DAYS, freq="D")

# Each asset: geometric random walk with a small drift
base_prices = [60000, 3500, 140, 0.45, 0.55, 0.15,
               35, 7, 14, 0.9, 5.5, 9]
drifts  = [0.003, 0.005, 0.008, -0.001, 0.001, 0.002,
           0.006, -0.002, 0.004, -0.003, 0.007, 0.009]
vols    = [0.025, 0.035, 0.05, 0.04, 0.03, 0.06,
           0.045, 0.04, 0.038, 0.05, 0.055, 0.048]

closes = pd.DataFrame(index=dates)
volumes = pd.DataFrame(index=dates)
for sym, p0, mu, sigma in zip(SYMBOLS, base_prices, drifts, vols):
    log_returns = np.random.normal(mu, sigma, DAYS)
    closes[sym]  = p0 * np.exp(np.cumsum(log_returns))
    volumes[sym] = np.random.uniform(5e6, 80e6, DAYS)  # quote volume in USD

print(f"Panel shape: {closes.shape}  ({len(SYMBOLS)} assets × {DAYS} days)")
closes.tail(3)

---
## 3 · Step-by-Step: Compute Momentum Scores

### 3.1  Single-Lookback Return

In [ ]:
def lookback_return(prices: pd.Series, lookback: int) -> float:
    """Return over the last `lookback` periods.
    
    r = P_t / P_{t-L} - 1
    """
    if len(prices) < lookback + 1 or prices.iloc[-(lookback + 1)] == 0:
        return float("nan")
    return (prices.iloc[-1] / prices.iloc[-(lookback + 1)]) - 1.0

# Demo: 3-day return for BTC
r3 = lookback_return(closes["BTCUSDT"], 3)
print(f"BTC 3-day return: {r3:+.4f}  ({r3*100:+.2f}%)")

### 3.2  Multi-Lookback Composite Score

Average returns across `[3, 5, 7]` day lookbacks — exactly what `rank_assets_by_momentum()` does.

In [ ]:
LOOKBACKS = [3, 5, 7]  # from strategy_params.yaml → momentum.lookback_days

def composite_momentum(prices: pd.Series, lookbacks: list[int]) -> float:
    """Mean of multi-lookback returns."""
    returns = [lookback_return(prices, lb) for lb in lookbacks]
    valid = [r for r in returns if not np.isnan(r)]
    return np.mean(valid) if valid else float("nan")

# Compute for every asset
raw_scores = {sym: composite_momentum(closes[sym], LOOKBACKS) for sym in SYMBOLS}
score_df = (
    pd.Series(raw_scores, name="composite_score")
    .to_frame()
    .sort_values("composite_score", ascending=False)
)
score_df["rank"] = range(1, len(score_df) + 1)
score_df.style.format({"composite_score": "{:+.4f}"})

---
## 4 · Quality Filters

Raw momentum scores are noisy.  Our production code applies **three filters** before ranking:

| Filter | Condition | Purpose |
|--------|-----------|--------|
| **RSI** | `RSI > 45` | Reject assets losing momentum |
| **EMA** | `price > EMA(20)` | Confirm uptrend |
| **Volume** | `quote_volume ≥ $10M` | Ensure liquidity |

An asset must pass **all three** to survive.

### 4.1  Why These Particular Filters?

- **RSI > 45** (not 50):  a small buffer avoids whipsawing around the midline.
- **Price > EMA-20**:  classic trend-following confirmation — we don't buy assets trading below their short-term average.
- **Volume ≥ $10M**:  thin markets have wide spreads and slippage; the bot must execute orders cleanly.

These thresholds live in `config/strategy_params.yaml`.

In [ ]:
# ── RSI calculation (from NB02 / production code) ─────────
def calculate_rsi(prices: pd.Series, period: int = 14) -> pd.Series:
    delta = prices.astype(float).diff()
    gains = delta.clip(lower=0.0)
    losses = (-delta).clip(lower=0.0)
    avg_gain = gains.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss = losses.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss_safe = avg_loss.mask(avg_loss == 0.0)
    rs = avg_gain / avg_loss_safe
    rsi = 100.0 - (100.0 / (1.0 + rs))
    rsi = rsi.mask((avg_loss == 0.0) & (avg_gain > 0.0), 100.0)
    rsi = rsi.mask((avg_gain == 0.0) & (avg_loss > 0.0), 0.0)
    rsi = rsi.mask((avg_gain == 0.0) & (avg_loss == 0.0), 50.0)
    return rsi.astype(float)

# ── Apply all three filters ───────────────────────────────
RSI_THRESHOLD  = 45.0           # from strategy_params.yaml
EMA_PERIOD     = 20
MIN_VOLUME_USD = 10_000_000.0

filter_records = []
for sym in SYMBOLS:
    price_series  = closes[sym].dropna()
    vol_series    = volumes[sym].dropna()
    
    current_price   = float(price_series.iloc[-1])
    current_ema     = float(price_series.ewm(span=EMA_PERIOD, adjust=False).mean().iloc[-1])
    current_rsi     = float(calculate_rsi(price_series).iloc[-1])
    current_volume  = float(vol_series.iloc[-1])
    
    pass_rsi    = current_rsi > RSI_THRESHOLD
    pass_ema    = current_price > current_ema
    pass_volume = current_volume >= MIN_VOLUME_USD
    
    filter_records.append({
        "symbol":  sym,
        "price":   current_price,
        "EMA-20":  current_ema,
        "RSI-14":  current_rsi,
        "vol_USD":  current_volume,
        "pass_rsi": pass_rsi,
        "pass_ema": pass_ema,
        "pass_vol": pass_volume,
        "ALL_PASS": pass_rsi and pass_ema and pass_volume,
    })

filter_df = pd.DataFrame(filter_records).set_index("symbol")

# Colour-code: green for pass, red for fail
def _colour_pass_fail(val):
    if isinstance(val, bool):
        return "color: green" if val else "color: red; font-weight: bold"
    return ""

filter_df.style.applymap(_colour_pass_fail).format({
    "price": "{:.4f}", "EMA-20": "{:.4f}",
    "RSI-14": "{:.1f}", "vol_USD": "{:,.0f}"
})

In [ ]:
# ── Survivors only ────────────────────────────────────────
survivors = filter_df[filter_df["ALL_PASS"]].index.tolist()
print(f"Survivors after quality filters: {len(survivors)} / {len(SYMBOLS)}")
print(survivors)

---
## 5 · Ranking & Min-Max Normalisation

After filtering, we:
1. Sort survivors by composite score.
2. Take the **top-N** (default 8 from `strategy_params.yaml`).
3. Normalise to $[0, 1]$:

$$
\text{norm}_i = \frac{s_i - s_{\min}}{s_{\max} - s_{\min}}
$$

When all scores are equal ($s_{\max} = s_{\min}$), every asset gets `normalized_score = 1.0`.

In [ ]:
TOP_N = 8  # from strategy_params.yaml → momentum.top_n_assets

# Composite scores for survivors only
survivor_scores = {
    sym: raw_scores[sym]
    for sym in survivors
    if not np.isnan(raw_scores.get(sym, float("nan")))
}
sorted_survivors = sorted(survivor_scores.items(),
                          key=lambda x: x[1], reverse=True)[:TOP_N]

# Min-max normalisation
scores_only = [s for _, s in sorted_survivors]
s_min, s_max = min(scores_only), max(scores_only)
span = s_max - s_min

ranking = []
for rank, (sym, score) in enumerate(sorted_survivors, 1):
    norm = 1.0 if span == 0 else (score - s_min) / span
    ranking.append({
        "rank": rank,
        "symbol": sym,
        "composite_score": score,
        "normalized_score": norm,
    })

ranking_df = pd.DataFrame(ranking).set_index("rank")
ranking_df.style.format({"composite_score": "{:+.5f}", "normalized_score": "{:.4f}"})

---
## 6 · Visualising Momentum

### 6.1  Bar Chart: Composite Scores

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Left: composite scores of ALL assets ----
ax = axes[0]
all_scores = pd.Series(raw_scores).sort_values(ascending=True)
colours = ["#2ecc71" if s > 0 else "#e74c3c" for s in all_scores]
all_scores.plot.barh(ax=ax, color=colours)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Composite Momentum Score")
ax.set_title("All Assets – Raw Composite Score")

# --- Right: normalised scores of TOP-N ----
ax2 = axes[1]
norm_series = ranking_df.set_index("symbol")["normalized_score"].sort_values()
norm_series.plot.barh(ax=ax2, color="#3498db")
ax2.set_xlabel("Normalised Score [0, 1]")
ax2.set_title(f"Top-{TOP_N} Survivors – Normalised Scores")

plt.tight_layout()
plt.show()

### 6.2  Price Trajectories of Top-N

In [ ]:
top_symbols = ranking_df["symbol"].tolist()

# Normalise all price series to start at 100 for comparison
norm_prices = closes[top_symbols].apply(lambda s: s / s.iloc[0] * 100)

fig, ax = plt.subplots(figsize=(14, 6))
for sym in top_symbols:
    ax.plot(norm_prices.index, norm_prices[sym], label=sym, lw=1.5)
ax.axhline(100, color="gray", ls="--", lw=0.8, label="Base = 100")
ax.set_ylabel("Normalised Price (start = 100)")
ax.set_title("Price Trajectories of Top Momentum Assets")
ax.legend(ncol=4, fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
plt.tight_layout()
plt.show()

---
## 7 · Sensitivity Analysis

How does the choice of lookback windows affect rankings?  Let's sweep different configurations.

In [ ]:
lookback_configs = {
    "Short [2,3,4]": [2, 3, 4],
    "Default [3,5,7]": [3, 5, 7],
    "Medium [5,10,15]": [5, 10, 15],
    "Long [7,14,21]": [7, 14, 21],
}

sensitivity = {}
for label, lbs in lookback_configs.items():
    scores = {
        sym: composite_momentum(closes[sym], lbs)
        for sym in survivors
    }
    ranked = sorted(scores, key=scores.get, reverse=True)
    sensitivity[label] = {sym: rank+1 for rank, sym in enumerate(ranked)}

sens_df = pd.DataFrame(sensitivity)
sens_df.index.name = "Symbol"
print("Rank by lookback configuration (lower = better):")
sens_df

**Observation:** assets with consistent momentum rank highly across *all* lookback windows — these are the most reliable candidates.

---
## 8 · Production Code Comparison

Let's call the **actual production function** `rank_assets_by_momentum()` from `bot/signals/momentum.py` and compare.

In [ ]:
from bot.signals.momentum import rank_assets_by_momentum, MomentumSignal

# Call production function with the same data
production_signals: list[MomentumSignal] = rank_assets_by_momentum(
    closes,
    volumes,
    lookback_periods=(3, 5, 7),
    rsi_period=14,
    rsi_threshold=45.0,
    ema_period=20,
    min_volume_usd=10_000_000.0,
    top_n_assets=8,
)

print(f"Production returned {len(production_signals)} signals\n")
prod_df = pd.DataFrame([
    {"rank": i+1, "symbol": s.symbol,
     "composite": s.composite_score,
     "normalised": s.normalized_score,
     "RSI": s.rsi, "EMA": s.ema, "volume": s.quote_volume}
    for i, s in enumerate(production_signals)
]).set_index("rank")

prod_df.style.format({
    "composite": "{:+.5f}", "normalised": "{:.4f}",
    "RSI": "{:.1f}", "EMA": "{:.4f}", "volume": "{:,.0f}"
})

In [ ]:
# ── Verify our scratch ranking matches production ─────────
our_ranking  = ranking_df["symbol"].tolist()
prod_ranking = [s.symbol for s in production_signals]

# They should agree on survivors (same filters, same formula)
shared = set(our_ranking) & set(prod_ranking)
print(f"Scratch ranking:     {our_ranking}")
print(f"Production ranking:  {prod_ranking}")
print(f"Overlap:             {len(shared)} / {max(len(our_ranking), len(prod_ranking))}")
print()

# Compare composite scores
for s in production_signals:
    if s.symbol in survivor_scores:
        ours = survivor_scores[s.symbol]
        diff = abs(ours - s.composite_score)
        print(f"  {s.symbol:10s}  ours={ours:+.6f}  prod={s.composite_score:+.6f}  Δ={diff:.2e}")

---
## 9 · Anatomy of `MomentumSignal`

The production code returns a **frozen dataclass**:

```python
@dataclass(frozen=True, slots=True)
class MomentumSignal:
    symbol: str
    composite_score: float   # average multi-lookback return
    normalized_score: float  # min-max normalised to [0, 1]
    price: float             # most recent close
    ema: float               # EMA-20 at latest bar
    rsi: float               # RSI-14 at latest bar
    quote_volume: float      # latest quote volume (USD)
```

**Design notes:**
- `frozen=True` prevents accidental mutation — signals are **immutable facts**.
- `slots=True` saves memory (important when polling hundreds of assets).
- The signal carries its own filter values (`rsi`, `ema`, `quote_volume`) so downstream modules can inspect *why* an asset was selected.

---
## 10 · Rolling Momentum: Time-Series View

Let's compute a rolling composite score for each asset over time to see how rankings shift.

In [ ]:
def rolling_composite(prices: pd.Series, lookbacks: list[int]) -> pd.Series:
    """Compute composite momentum score at every bar."""
    min_lb = max(lookbacks) + 1
    scores = pd.Series(index=prices.index, dtype=float)
    for i in range(min_lb, len(prices)):
        window = prices.iloc[:i+1]
        scores.iloc[i] = composite_momentum(window, lookbacks)
    return scores

# Compute for top-4 assets
top4 = ranking_df["symbol"].tolist()[:4]
fig, ax = plt.subplots(figsize=(14, 5))
for sym in top4:
    rolling = rolling_composite(closes[sym], LOOKBACKS)
    ax.plot(rolling.dropna(), label=sym, lw=1.5)
ax.axhline(0, color="gray", ls="--", lw=0.8)
ax.set_ylabel("Composite Momentum Score")
ax.set_title("Rolling Momentum Score Over Time")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
plt.tight_layout()
plt.show()

---
## 11 · Simple Momentum Backtest

Let's build a minimal backtest: every day, go long the top-3 momentum assets (equal weight) and measure cumulative returns.

In [ ]:
# ── Naïve daily momentum backtest ─────────────────────────
MIN_HISTORY = max(LOOKBACKS) + 1
TOP_K = 3

daily_returns = closes.pct_change()
strategy_returns = []

for i in range(MIN_HISTORY, len(closes)):
    # At close of day i, rank using data up to day i
    window = closes.iloc[:i+1]
    scores = {}
    for sym in SYMBOLS:
        s = composite_momentum(window[sym], LOOKBACKS)
        if not np.isnan(s):
            # Simple filter: only positive momentum
            if s > 0:
                scores[sym] = s
    
    # Top-K by momentum
    selected = sorted(scores, key=scores.get, reverse=True)[:TOP_K]
    
    if selected and i + 1 < len(closes):
        # Next-day return (equal weight)
        next_ret = daily_returns.iloc[i + 1][selected].mean()
        strategy_returns.append({
            "date": closes.index[i + 1],
            "return": next_ret,
            "holdings": selected,
        })

strat_df = pd.DataFrame(strategy_returns).set_index("date")
strat_df["cumulative"] = (1 + strat_df["return"]).cumprod()

# Benchmark: equal-weight all 12 assets
benchmark_ret = daily_returns.iloc[MIN_HISTORY+1:].mean(axis=1)
benchmark_cum = (1 + benchmark_ret).cumprod()

# Plot
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(strat_df.index, strat_df["cumulative"], label=f"Momentum Top-{TOP_K}", lw=2)
ax.plot(benchmark_cum.index, benchmark_cum, label="Equal-Weight Benchmark", lw=2, ls="--")
ax.set_ylabel("Cumulative Return")
ax.set_title("Momentum Strategy vs Equal-Weight Benchmark")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
plt.tight_layout()
plt.show()

# Stats
total_ret  = strat_df["cumulative"].iloc[-1] - 1
ann_vol    = strat_df["return"].std() * np.sqrt(365)
sharpe     = (strat_df["return"].mean() / strat_df["return"].std()) * np.sqrt(365) if strat_df["return"].std() > 0 else 0
print(f"\nMomentum strategy: total return = {total_ret:+.2%}, ann. vol = {ann_vol:.2%}, Sharpe ≈ {sharpe:.2f}")
bench_total = benchmark_cum.iloc[-1] - 1
print(f"Benchmark:         total return = {bench_total:+.2%}")

---
## 12 · Key Takeaways

| Concept | Detail |
|---------|--------|
| **Multi-lookback** | Averaging 3/5/7-day returns reduces noise |
| **Quality filters** | RSI > 45, Price > EMA-20, Volume ≥ $10M |
| **Min-max normalisation** | Maps scores to [0,1] for downstream weight allocation |
| **MomentumSignal** | Frozen dataclass carries score + all filter values |
| **Config-driven** | All thresholds come from `strategy_params.yaml` |

### The Momentum Pipeline at a Glance

```
Universe (N assets)
  │
  ├── Compute composite_score for each
  │     └── avg of lookback returns [3, 5, 7]
  │
  ├── Filter: RSI > 45
  ├── Filter: Price > EMA(20)
  ├── Filter: Volume ≥ $10M
  │
  ├── Sort by composite_score (desc)
  ├── Take top-N
  └── Min-max normalise → MomentumSignal[]
```

---
## 🔬 Exercises

1. **RSI threshold sweep:** Change `RSI_THRESHOLD` from 30 to 60 in steps of 5 and plot how many assets survive at each level.  What is the optimal threshold for maximising the Sharpe ratio of the naïve backtest?

2. **Lookback selection:** Try `[1, 3, 5]` (more reactive) and `[7, 14, 21]` (slower).  How does the turnover (number of rank changes per day) differ?

3. **Momentum crash:** Momentum strategies are vulnerable to **sudden reversals** (see "momentum crashes" literature).  Modify the backtest to add a stop-loss: if a holding drops more than 3% in a single day, exit immediately.  Does it improve the Sharpe?

4. **Volume filter ablation:** Remove the volume filter entirely.  Does the backtest performance change?  Why or why not with synthetic data vs real data?

---
## ✅ Knowledge Check

1. Why does the bot average returns over multiple lookback periods instead of using a single one?
2. What happens to `normalized_score` when all surviving assets have the same composite score?
3. Why is `MomentumSignal` a **frozen** dataclass?
4. Name the three quality filters and explain the purpose of each.
5. How would you modify the code to prefer assets with *improving* momentum (positive second derivative)?

---
## 🔗 Next

**[NB05 — Mean Reversion & Pairs Trading →](05_Mean_Reversion_and_Pairs_Trading.ipynb)**

We'll explore the *opposite* philosophy — buying assets that have fallen too far and are likely to revert to their mean — and learn how to trade **pairs** using cointegration.